<a href="https://colab.research.google.com/github/tracyhann/bsl-data-tools/blob/main/study_folder_initializer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🔗 Connect to Google Drive {"run":"auto","display-mode":"form"}
# Colab cell 1: install + authenticate
!pip install -q google-api-python-client google-auth google-auth-httplib2

from google.colab import auth
auth.authenticate_user()

import google.auth
import logging
from googleapiclient.discovery import build

creds, project_id = google.auth.default()
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

drive_service = build("drive", "v3", credentials=creds)

In [ ]:
# @title 🧬 Initiate study folder from template {"run":"auto","vertical-output":true,"display-mode":"form"}
# @markdown Creates a new study folder by copying the full STUDY_IRB template folder.
# @markdown Root folder naming: `STUDY (IRB1/IRB2/...)`
# @markdown Internal file/folder naming replaces `STUDY` and `IRB` placeholders.
# @markdown Example: STUDY=`BRAINS`, IRB=`54909, 58807`
# @markdown Root: `BRAINS (54909/58807)`
# @markdown Internal placeholder: `STUDY_IRB` -> `BRAINS_54909-58807`

DESTINATION_PARENT_FOLDER_URL = "https://drive.google.com/drive/folders/15n8HHni7pGxxSxza8They6RFe0j5FU-j" # @param {"type":"string","placeholder":"Paste destination Google Drive folder link here"}

STUDY = "aTBS" # @param {"type":"string","placeholder":"Ex: BRAINS"}
IRB = "49486" # @param {"type":"string","placeholder":"Ex: 54909,58807"}

DRY_RUN = False # @param {"type":"boolean"}
FAIL_IF_ROOT_EXISTS = True # @param {"type":"boolean"}
COPY_SHORTCUT_TARGETS = False # @param {"type":"boolean"}

# Fixed template root.
TEMPLATE_FOLDER_URL = "https://drive.google.com/drive/folders/1OLGHgxPg9UsBDbOH6vgSjiS6dUW0lBCF?usp=sharing"


import re
import csv
import time
import random
import logging
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from typing import Optional

from IPython.display import HTML, display
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


# =========================
# Setup
# =========================

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"

auth.authenticate_user()
credentials, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)


# =========================
# User input formatting
# =========================

def parse_irbs(irb_raw: str) -> list[str]:
    """
    '54909, 58807, 60001' -> ['54909', '58807', '60001']
    """
    parts = [
        re.sub(r"\s+", "", x.strip())
        for x in irb_raw.split(",")
        if x.strip()
    ]

    if not parts:
        raise ValueError("IRB cannot be empty.")

    return parts


def clean_study(study_raw: str) -> str:
    study = re.sub(r"\s+", " ", study_raw.strip())

    if not study:
        raise ValueError("STUDY cannot be empty.")

    return study


STUDY_VALUE = clean_study(STUDY)
IRB_LIST = parse_irbs(IRB)

# For placeholders inside copied files/folders/docs.
IRB_HYPHEN = "-".join(IRB_LIST)

# For root folder naming only.
IRB_SLASH = "/".join(IRB_LIST)

# Root exception:
#   STUDY (IRB1/IRB2/...)
DEST_ROOT_NAME = f"{STUDY_VALUE} ({IRB_SLASH})"

# Placeholder value:
#   STUDY_IRB -> STUDY_IRB1-IRB2-...
STUDY_IRB_VALUE = f"{STUDY_VALUE}_{IRB_HYPHEN}"


def apply_placeholders(name: str) -> str:
    """
    Replaces placeholders in copied item names.

    Supported patterns:
      STUDY_IRB      -> <STUDY>_<IRB1-IRB2>
      {STUDY_IRB}    -> <STUDY>_<IRB1-IRB2>
      STUDY          -> <STUDY>
      {STUDY}        -> <STUDY>
      IRB            -> <IRB1-IRB2>
      {IRB}          -> <IRB1-IRB2>

    Root folder uses separate exception naming and does not use this function.
    """
    new = name

    # Combined placeholder first.
    new = new.replace("{STUDY_IRB}", STUDY_IRB_VALUE)
    new = new.replace("STUDY_IRB", STUDY_IRB_VALUE)

    # Braced placeholders.
    new = new.replace("{STUDY}", STUDY_VALUE)
    new = new.replace("{IRB}", IRB_HYPHEN)

    # Standalone uppercase placeholders.
    new = re.sub(r"\bSTUDY\b", STUDY_VALUE, new)
    new = re.sub(r"\bIRB\b", IRB_HYPHEN, new)

    return new


# =========================
# Drive helpers
# =========================

def extract_drive_id(url_or_id: str) -> str:
    if not url_or_id:
        raise ValueError("Missing Google Drive URL or ID.")

    text = url_or_id.strip()

    if re.fullmatch(r"[a-zA-Z0-9_-]{20,}", text):
        return text

    parsed = urlparse(text)

    query_id = parse_qs(parsed.query).get("id")
    if query_id:
        return query_id[0]

    patterns = [
        r"/folders/([a-zA-Z0-9_-]+)",
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"/document/d/([a-zA-Z0-9_-]+)",
        r"/spreadsheets/d/([a-zA-Z0-9_-]+)",
        r"/presentation/d/([a-zA-Z0-9_-]+)",
        r"/forms/d/([a-zA-Z0-9_-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    raise ValueError(f"Could not extract Google Drive ID from: {url_or_id}")


def drive_query_quote(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"


def should_retry(error: HttpError) -> bool:
    status = getattr(error.resp, "status", None)

    try:
        content = error.content.decode("utf-8", errors="ignore")
    except Exception:
        content = ""

    if status in {429, 500, 502, 503, 504}:
        return True

    if status == 403 and any(
        reason in content
        for reason in [
            "rateLimitExceeded",
            "userRateLimitExceeded",
            "backendError",
        ]
    ):
        return True

    return False


def execute(request, label: str = "Drive API request"):
    max_retries = 6

    for attempt in range(max_retries):
        try:
            return request.execute()
        except HttpError as e:
            if attempt < max_retries - 1 and should_retry(e):
                wait = min(60, 2 ** attempt + random.random())
                print(f"⏳ Retry {attempt + 1}/{max_retries} after {wait:.1f}s: {label}")
                time.sleep(wait)
                continue
            raise


def get_metadata(file_id: str) -> dict:
    return execute(
        drive_service.files().get(
            fileId=file_id,
            fields="id,name,mimeType,webViewLink,shortcutDetails",
            supportsAllDrives=True,
        ),
        label=f"get metadata {file_id}",
    )


def list_children(folder_id: str) -> list[dict]:
    children = []
    page_token = None

    while True:
        response = execute(
            drive_service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields=(
                    "nextPageToken,"
                    "files(id,name,mimeType,webViewLink,shortcutDetails)"
                ),
                pageSize=1000,
                pageToken=page_token,
                orderBy="folder,name",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                corpora="allDrives",
            ),
            label=f"list children {folder_id}",
        )

        children.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return children


def find_existing_child(
    parent_id: str,
    name: str,
    mime_type: Optional[str] = None,
) -> list[dict]:
    q_parts = [
        f"'{parent_id}' in parents",
        "trashed = false",
        f"name = {drive_query_quote(name)}",
    ]

    if mime_type:
        q_parts.append(f"mimeType = {drive_query_quote(mime_type)}")

    response = execute(
        drive_service.files().list(
            q=" and ".join(q_parts),
            fields="files(id,name,mimeType,webViewLink)",
            pageSize=10,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
            corpora="allDrives",
        ),
        label=f"find existing child {name}",
    )

    return response.get("files", [])


def make_link(name: str, url: Optional[str]) -> str:
    safe_name = name

    if not url:
        return safe_name

    return f'<a href="{url}" target="_blank">{safe_name}</a>'


# =========================
# Copy helpers
# =========================

copy_log = []


def log_action(
    action: str,
    source_path: str,
    dest_path: str,
    source_id: str = "",
    dest_id: str = "",
    mime_type: str = "",
    error: str = "",
):
    copy_log.append(
        {
            "action": action,
            "source_path": source_path,
            "dest_path": dest_path,
            "source_id": source_id,
            "dest_id": dest_id,
            "mime_type": mime_type,
            "error": error,
        }
    )


def create_folder(
    name: str,
    parent_id: str,
    source_path: str,
    dest_path: str,
) -> dict:
    """
    Drive cannot copy folders directly.
    This creates the destination folder container.
    Actual files/docs inside are copied item-by-item from templates.
    """
    if DRY_RUN:
        print(f"📁 [DRY] create folder: {dest_path}")
        log_action("dry_create_folder", source_path, dest_path, mime_type=FOLDER_MIME)
        return {
            "id": f"DRY_FOLDER::{dest_path}",
            "name": name,
            "webViewLink": "",
        }

    created = execute(
        drive_service.files().create(
            body={
                "name": name,
                "mimeType": FOLDER_MIME,
                "parents": [parent_id],
            },
            fields="id,name,webViewLink",
            supportsAllDrives=True,
        ),
        label=f"create folder {dest_path}",
    )

    print(f"📁 created folder: {dest_path}")
    log_action(
        "create_folder",
        source_path,
        dest_path,
        dest_id=created.get("id", ""),
        mime_type=FOLDER_MIME,
    )

    return created


def copy_file(
    file_id: str,
    new_name: str,
    parent_id: str,
    source_path: str,
    dest_path: str,
    mime_type: str,
) -> Optional[dict]:
    """
    Copies the actual template file/doc/sheet/slide and renames the copy.
    """
    if DRY_RUN:
        print(f"📄 [DRY] copy file/doc: {source_path} -> {dest_path}")
        log_action("dry_copy_file", source_path, dest_path, source_id=file_id, mime_type=mime_type)
        return None

    copied = execute(
        drive_service.files().copy(
            fileId=file_id,
            body={
                "name": new_name,
                "parents": [parent_id],
            },
            fields="id,name,webViewLink",
            supportsAllDrives=True,
        ),
        label=f"copy file {source_path}",
    )

    print(f"📄 copied file/doc: {dest_path}")
    log_action(
        "copy_file",
        source_path,
        dest_path,
        source_id=file_id,
        dest_id=copied.get("id", ""),
        mime_type=mime_type,
    )

    return copied


def create_shortcut(
    shortcut_item: dict,
    new_name: str,
    parent_id: str,
    source_path: str,
    dest_path: str,
) -> Optional[dict]:
    target_id = shortcut_item.get("shortcutDetails", {}).get("targetId")

    if not target_id:
        print(f"⚠️ shortcut has no target, skipping: {source_path}")
        log_action(
            "skip_shortcut_no_target",
            source_path,
            dest_path,
            source_id=shortcut_item.get("id", ""),
            mime_type=SHORTCUT_MIME,
        )
        return None

    if DRY_RUN:
        print(f"🔗 [DRY] recreate shortcut: {source_path} -> {dest_path}")
        log_action(
            "dry_create_shortcut",
            source_path,
            dest_path,
            source_id=shortcut_item.get("id", ""),
            mime_type=SHORTCUT_MIME,
        )
        return None

    created = execute(
        drive_service.files().create(
            body={
                "name": new_name,
                "mimeType": SHORTCUT_MIME,
                "parents": [parent_id],
                "shortcutDetails": {
                    "targetId": target_id,
                },
            },
            fields="id,name,webViewLink",
            supportsAllDrives=True,
        ),
        label=f"create shortcut {source_path}",
    )

    print(f"🔗 recreated shortcut: {dest_path}")
    log_action(
        "create_shortcut",
        source_path,
        dest_path,
        source_id=shortcut_item.get("id", ""),
        dest_id=created.get("id", ""),
        mime_type=SHORTCUT_MIME,
    )

    return created


def copy_tree(
    source_folder_id: str,
    dest_folder_id: str,
    source_path: str,
    dest_path: str,
    visited: Optional[set[str]] = None,
):
    """
    Recursively copies template contents into destination folder.
    """
    if visited is None:
        visited = set()

    if source_folder_id in visited:
        print(f"↩️ loop detected, skipping: {source_path}")
        log_action("skip_loop", source_path, dest_path, source_id=source_folder_id, dest_id=dest_folder_id)
        return

    visited = set(visited)
    visited.add(source_folder_id)

    children = list_children(source_folder_id)

    if not children:
        print(f"∅ empty folder preserved: {dest_path}")
        log_action("empty_folder", source_path, dest_path, source_id=source_folder_id, dest_id=dest_folder_id)
        return

    for item in children:
        old_name = item["name"]
        new_name = apply_placeholders(old_name)

        item_id = item["id"]
        mime_type = item["mimeType"]

        source_child_path = f"{source_path}/{old_name}".strip("/")
        dest_child_path = f"{dest_path}/{new_name}".strip("/")

        try:
            if mime_type == FOLDER_MIME:
                new_folder = create_folder(
                    name=new_name,
                    parent_id=dest_folder_id,
                    source_path=source_child_path,
                    dest_path=dest_child_path,
                )

                copy_tree(
                    source_folder_id=item_id,
                    dest_folder_id=new_folder["id"],
                    source_path=source_child_path,
                    dest_path=dest_child_path,
                    visited=visited,
                )

            elif mime_type == SHORTCUT_MIME:
                shortcut_details = item.get("shortcutDetails", {})
                target_id = shortcut_details.get("targetId")
                target_mime = shortcut_details.get("targetMimeType")

                if COPY_SHORTCUT_TARGETS and target_id:
                    print(f"🔗 dereferencing shortcut: {source_child_path}")

                    if target_mime == FOLDER_MIME:
                        new_folder = create_folder(
                            name=new_name,
                            parent_id=dest_folder_id,
                            source_path=source_child_path,
                            dest_path=dest_child_path,
                        )

                        copy_tree(
                            source_folder_id=target_id,
                            dest_folder_id=new_folder["id"],
                            source_path=source_child_path,
                            dest_path=dest_child_path,
                            visited=visited,
                        )
                    else:
                        copy_file(
                            file_id=target_id,
                            new_name=new_name,
                            parent_id=dest_folder_id,
                            source_path=source_child_path,
                            dest_path=dest_child_path,
                            mime_type=target_mime or SHORTCUT_MIME,
                        )
                else:
                    create_shortcut(
                        shortcut_item=item,
                        new_name=new_name,
                        parent_id=dest_folder_id,
                        source_path=source_child_path,
                        dest_path=dest_child_path,
                    )

            else:
                copy_file(
                    file_id=item_id,
                    new_name=new_name,
                    parent_id=dest_folder_id,
                    source_path=source_child_path,
                    dest_path=dest_child_path,
                    mime_type=mime_type,
                )

        except Exception as e:
            print(f"❌ failed: {source_child_path} -> {dest_child_path}")
            print(f"   {type(e).__name__}: {e}")
            log_action(
                "error",
                source_child_path,
                dest_child_path,
                source_id=item_id,
                mime_type=mime_type,
                error=str(e),
            )


def write_log_csv() -> str:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    mode = "dry_run" if DRY_RUN else "copy"
    path = f"/content/study_folder_initiation_{mode}_{timestamp}.csv"

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "action",
                "source_path",
                "dest_path",
                "source_id",
                "dest_id",
                "mime_type",
                "error",
            ],
        )
        writer.writeheader()
        writer.writerows(copy_log)

    return path


# =========================
# Run
# =========================

if not DESTINATION_PARENT_FOLDER_URL.strip():
    raise ValueError("Paste the destination parent Google Drive folder link.")

template_folder_id = extract_drive_id(TEMPLATE_FOLDER_URL)
dest_parent_id = extract_drive_id(DESTINATION_PARENT_FOLDER_URL)

template_meta = get_metadata(template_folder_id)
dest_parent_meta = get_metadata(dest_parent_id)

if template_meta["mimeType"] != FOLDER_MIME:
    raise ValueError(f"Template root is not a folder: {template_meta['name']}")

if dest_parent_meta["mimeType"] != FOLDER_MIME:
    raise ValueError(f"Destination parent is not a folder: {dest_parent_meta['name']}")

print("Template root:", template_meta["name"])
print("Destination parent:", dest_parent_meta["name"])
print("New study root:", DEST_ROOT_NAME)
print("STUDY placeholder value:", STUDY_VALUE)
print("IRB placeholder value:", IRB_HYPHEN)
print("Mode:", "DRY RUN, no files/folders will be created" if DRY_RUN else "REAL COPY")
print("Shortcut behavior:", "copy shortcut targets" if COPY_SHORTCUT_TARGETS else "recreate shortcuts as shortcuts")
print()

if FAIL_IF_ROOT_EXISTS and not DRY_RUN:
    existing = find_existing_child(
        parent_id=dest_parent_id,
        name=DEST_ROOT_NAME,
        mime_type=FOLDER_MIME,
    )

    if existing:
        links = "\n".join(f"- {x['name']}: {x.get('webViewLink', x['id'])}" for x in existing)
        raise RuntimeError(
            f"A folder named {DEST_ROOT_NAME!r} already exists in the destination.\n"
            f"Delete/rename it, or set FAIL_IF_ROOT_EXISTS = False.\n\n{links}"
        )

# Root folder exception:
# root name is STUDY (IRB1/IRB2/...)
dest_root = create_folder(
    name=DEST_ROOT_NAME,
    parent_id=dest_parent_id,
    source_path=template_meta["name"],
    dest_path=f'{dest_parent_meta["name"]}/{DEST_ROOT_NAME}',
)

log_action(
    "dry_create_root_folder" if DRY_RUN else "create_root_folder",
    template_meta["name"],
    DEST_ROOT_NAME,
    source_id=template_folder_id,
    dest_id=dest_root.get("id", ""),
    mime_type=FOLDER_MIME,
)

copy_tree(
    source_folder_id=template_folder_id,
    dest_folder_id=dest_root["id"],
    source_path=template_meta["name"],
    dest_path=DEST_ROOT_NAME,
)

log_path = write_log_csv()

print()
print("✅ Done.")
print("Log CSV:", log_path)

if DRY_RUN:
    print("Review the plan above. Then set DRY_RUN = False and rerun to actually create the study folder.")
else:
    display(HTML(f'Created study folder: <a href="{dest_root.get("webViewLink")}" target="_blank">{DEST_ROOT_NAME}</a>'))